# Synthetic Fintech AFC Transactions and Data Quality Lab

Generate 1,000,000 fictional records for testing Anti-Financial-Crime (AFC) data ingestion and quality controls. Preserve the exact 12-column reference schema. All valid underlying dates, including transaction-ID date prefixes, are between **2025-01-01 and 2026-09-30**, inclusive. Intentionally invalid dates are labeled as injected noise in the generation summary.

PayPal, Amazon, Venmo, and Payoneer are used only as simulated platform labels prefixed with SYNTHETIC. No records describe actual customers or activity at these companies. Product weights, payment patterns, amounts, and noise rates are illustrative test settings, not actual platform capabilities or estimates of financial crime. Data-quality anomalies are not evidence of illicit activity.

Run all cells using Python 3.11 in VS Code/Jupyter. The notebook uses only the standard library; the notebook environment may require ipykernel. No real data is read and no network access is used. The raw CSV retains deliberate errors so downstream systems can be tested. The quality pass demonstrates normalization and quarantine decisions without inventing corrected dates or amounts.

The existing raw CSV is intentionally replaced by this revised dataset. Set OVERWRITE=False to protect it on future runs. Enable WRITE_CLEAN_OUTPUTS only if you also want separate clean and quarantine CSVs. A quality-report JSON file is always produced. CSV identifiers must be imported as text in Excel.


## Schema and test coverage

The columns remain: `trxn_id`, `lvl_2_product_txt`, `amount`, `value_date`, `orig_raw_account_num`, `bene_raw_account_num`, `orig_raw_name`, `bene_raw_name`, `orig_clean_name`, `bene_clean_name`, `orig_cntry_code`, `bene_cntry_code`.

- Products: Wires, Domestic ACH, International ACH, Card Payment, Wallet Transfer, P2P Transfer, Merchant Settlement, and Cross-Border Remittance. `Internation ACH` is injected as a known spelling error and standardized to `International ACH`.
- Parties: fictional individuals, merchants, businesses, and simulated fintech platform accounts. Scenarios include wallet funding, P2P payments, marketplace payouts, merchant settlements, and cross-border payments. The fixed schema does not contain currency, timestamps, customer profiles, or crime labels; these are not inferred.
- Accounts: numeric identifiers with leading zeros, currency-suffixed identifiers, and deliberately invalid-checksum IBAN-shaped test identifiers. They are not usable bank accounts.
- Amounts: positive integer minor units with smaller retail amounts, larger settlements, and occasional high-value payments. No risk classification is inferred from amount alone.
- Names: raw names can be mixed case or legitimately absent; clean names are uppercase and explicitly synthetic. Country codes cover domestic and cross-border corridors.

A noise mutation is applied to a configurable fraction of rows, selected uniformly from the categories below. Independently, MISSING_ROW_RATE selects rows for blank-value injection. For each selected row, one to MAX_MISSING_FIELDS existing columns are sampled without replacement and set to the empty string. All 12 existing columns are eligible. A row may contain both noise and missing values; blank injection occurs last and may replace a previously injected noise value. Consequently, injection counts and observed raw-value counts are reported separately. A mutation may create multiple quality findings. Duplicate IDs are intentional; unique-ID validation must distinguish these from clean data.

| Noise | Handling |
|---|---|
| Date `99999999`, impossible date, or missing date | Quarantine; do not guess a replacement |
| Date `YYYY/MM/DD` or `YYYY-MM-DD` | Parse strictly and normalize to `YYYYMMDD` |
| Product typo, mixed case, or extra whitespace | Map only approved aliases to canonical labels |
| Amount with thousands separators | Remove separators only when grouping is valid |
| Nonnumeric, negative, or missing amount | Quarantine under this positive-payment test contract |
| Missing ID, repeated ID, or missing account | Quarantine; preserve the original row |
| Country lowercase | Convert to uppercase |
| Country `97`, `HH`, `ZZ`, or missing country | Quarantine |
| Account surrounding whitespace | Trim without removing leading zeros |
| Name casing/whitespace or missing clean name | Normalize text, or quarantine if required name is absent |

The schema remains exactly the original 12 columns. There are no city columns, so no city fields or city placeholders are introduced. Country noise is injected into either orig_cntry_code or bene_cntry_code. Empty CSV fields are actual missing values, not strings such as "None", "NaN", or "NULL". Optional raw-name blanks are accepted; missing required values are quarantined without guessing replacements.

Zero/negative amounts might represent real refunds or reversals in another business contract; this notebook intentionally uses a positive-payment contract. Country validation uses the explicitly configured test universe, not a complete ISO-country validator. Repeated IDs are flagged after their first physical occurrence, even if the earlier row has another error; production deduplication needs an agreed policy.


In [1]:
from pathlib import Path
from datetime import date, timedelta, datetime
from collections import Counter
import csv
import json
import os
import random
import re
import sqlite3
import tempfile
import time

ROW_COUNT = 1_000_000
SEED = 20260930
NOISE_RATE = 0.08
MISSING_ROW_RATE = 0.03  # Independent probability of selecting a row for blank injection.
MAX_MISSING_FIELDS = 3  # Select between one and this many existing fields per selected row.
OVERWRITE = True
WRITE_CLEAN_OUTPUTS = False
PROJECT_DIR = None  # Example: Path(r"C:\FCO-Review-main")
HEADERS = ["trxn_id", "lvl_2_product_txt", "amount", "value_date",
           "orig_raw_account_num", "bene_raw_account_num", "orig_raw_name", "bene_raw_name",
           "orig_clean_name", "bene_clean_name", "orig_cntry_code", "bene_cntry_code"]
if PROJECT_DIR is None:
    PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / "convert.py").is_file() and (p / "mapping.py").is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError("Set PROJECT_DIR to the project folder.")
PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
if not PROJECT_DIR.is_dir():
    raise ValueError("PROJECT_DIR must exist.")
if not isinstance(ROW_COUNT, int) or isinstance(ROW_COUNT, bool) or not 1 <= ROW_COUNT <= 9_999_999:
    raise ValueError("ROW_COUNT must be between 1 and 9,999,999.")
if not 0 <= NOISE_RATE <= 1:
    raise ValueError("NOISE_RATE must be between zero and one.")
if not 0 <= MISSING_ROW_RATE <= 1:
    raise ValueError("MISSING_ROW_RATE must be between zero and one.")
if not isinstance(MAX_MISSING_FIELDS, int) or isinstance(MAX_MISSING_FIELDS, bool) or not 1 <= MAX_MISSING_FIELDS <= len(HEADERS):
    raise ValueError("MAX_MISSING_FIELDS must be between one and the number of existing columns.")
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "synthetic_wire_transactions_1m.csv"
REPORT_FILE = OUTPUT_DIR / "synthetic_afc_quality_report.json"
CLEAN_FILE = OUTPUT_DIR / "synthetic_afc_clean.csv"
QUARANTINE_FILE = OUTPUT_DIR / "synthetic_afc_quarantine.csv"
START_DATE = date(2025, 1, 1)
END_DATE = date(2026, 9, 30)
DATES = [(START_DATE + timedelta(days=i)).strftime("%Y%m%d")
         for i in range((END_DATE - START_DATE).days + 1)]
COUNTRIES = ("US", "GB", "DE", "FR", "CA", "AU", "JP", "SG", "NL", "CH", "SE", "NZ", "IN", "MX", "BR")
PRODUCTS = ("Wires", "Domestic ACH", "International ACH", "Card Payment", "Wallet Transfer",
            "P2P Transfer", "Merchant Settlement", "Cross-Border Remittance")
WEIGHTS = (10, 20, 8, 15, 15, 12, 12, 8)
NOISE_TYPES = ("date_sentinel", "date_impossible", "date_missing", "date_slash", "date_dash",
               "product_typo", "product_spacing", "amount_grouping", "amount_text", "amount_negative",
               "amount_missing", "id_missing", "id_duplicate", "account_missing", "account_spacing",
               "country_lowercase", "country_invalid", "country_numeric_97", "country_alpha_hh",
               "country_missing", "name_spacing", "name_missing")
print(f"Output: {OUTPUT_FILE}")
print(f"Records: {ROW_COUNT:,}; valid dates: {START_DATE} through {END_DATE}; noise rate: {NOISE_RATE:.1%}")


Output: /Users/felicia/Desktop/Felicia/DB_work/FCO column mapping/output/synthetic_wire_transactions_1m.csv
Records: 1,000,000; valid dates: 2025-01-01 through 2026-09-30; noise rate: 8.0%


In [2]:
PLATFORMS = ("PAYPAL", "AMAZON", "VENMO", "PAYONEER")
FIRST = ("ELARA", "TAVIN", "NERIN", "ALORA", "DEVAN", "MIREL", "SOREN", "LYRA")
LAST = ("ASTER", "QUILL", "VALE", "SOLEN", "ROWAN", "FEN", "ORIN", "LARK")
BRANDS = ("AURORA", "COBALT", "QUARTZ", "CEDAR", "LUNAR", "EMBER")
SECTORS = ("MARKET", "LOGISTICS", "DIGITAL", "RETAIL", "EXPORTS", "SERVICES")

def party_name(rng, kind, seq, role):
    if kind == "platform":
        name = f"{rng.choice(PLATFORMS)} TEST WALLET"
    elif kind == "person":
        name = f"{rng.choice(FIRST)} {rng.choice(LAST)}"
    else:
        name = f"{rng.choice(BRANDS)} {rng.choice(SECTORS)} LTD"
    return f"SYNTHETIC {name} {role}{seq:07d}"

def account(rng, seq, role):
    digit = "1" if role == "O" else "2"
    style = rng.randrange(3)
    if style == 0:
        return f"0{digit}{seq:07d}{rng.randrange(100000):05d}"
    if style == 1:
        return f"GB00SYNX{digit}{seq:09d}{rng.randrange(100):02d}"
    return f"99{digit}{seq:010d}{rng.choice(('USD', 'EUR', 'GBP'))}"

def clean_record(rng, seq):
    product = rng.choices(PRODUCTS, weights=WEIGHTS, k=1)[0]
    if product in ("Merchant Settlement", "International ACH"):
        roles, bounds = ("platform", "business"), (10_000, 50_000_000)
    elif product == "P2P Transfer":
        roles, bounds = ("person", "person"), (100, 500_000)
    elif product in ("Wallet Transfer", "Domestic ACH"):
        roles, bounds = ("person", "platform"), (100, 2_000_000)
    elif product == "Card Payment":
        roles, bounds = ("person", "business"), (100, 1_000_000)
    else:
        roles, bounds = ("business", "business"), (1_000, 20_000_000)
    orig = rng.choice(COUNTRIES)
    if product in ("International ACH", "Cross-Border Remittance"):
        bene = rng.choice([country for country in COUNTRIES if country != orig])
    elif product in ("Domestic ACH", "P2P Transfer", "Wallet Transfer"):
        bene = orig
    else:
        bene = orig if rng.random() < 0.6 else rng.choice(COUNTRIES)
    day = rng.choice(DATES)
    names = [party_name(rng, roles[0], seq, "O"), party_name(rng, roles[1], seq, "B")]
    raw_names = [name.title() if rng.random() < 0.65 else "" for name in names]
    amount = rng.randint(*bounds)
    if rng.random() < 0.005:
        amount = rng.randint(50_000_001, 200_000_000)
    return [f"{day}{rng.choice(('C', 'S'))}{seq:07d}", product, str(amount), day,
            account(rng, seq, "O"), account(rng, seq, "B"), *raw_names, *names, orig, bene]

def inject_noise(row, kind, previous_id, noise_rng):
    if kind == "date_sentinel": row[3] = "99999999"
    elif kind == "date_impossible": row[3] = "20260230"
    elif kind == "date_missing": row[3] = ""
    elif kind in ("date_slash", "date_dash"):
        separator = "/" if kind == "date_slash" else "-"
        row[3] = separator.join((row[3][:4], row[3][4:6], row[3][6:]))
    elif kind == "product_typo": row[1] = "Internation ACH"
    elif kind == "product_spacing": row[1] = "  " + row[1].lower() + "  "
    elif kind == "amount_grouping": row[2] = f"{int(row[2]):,}"
    elif kind == "amount_text": row[2] = "NOT_AVAILABLE"
    elif kind == "amount_negative": row[2] = "-" + row[2]
    elif kind == "amount_missing": row[2] = ""
    elif kind == "id_missing": row[0] = ""
    elif kind == "id_duplicate": row[0] = previous_id
    elif kind == "account_missing": row[4] = ""
    elif kind == "account_spacing": row[5] = "  " + row[5] + "  "
    elif kind in ("country_lowercase", "country_invalid", "country_numeric_97", "country_alpha_hh", "country_missing"):
        column = noise_rng.choice((10, 11))
        if kind == "country_lowercase": row[column] = row[column].lower()
        elif kind == "country_invalid": row[column] = "ZZ"
        elif kind == "country_numeric_97": row[column] = "97"
        elif kind == "country_alpha_hh": row[column] = "HH"
        else: row[column] = ""
    elif kind == "name_spacing": row[8] = "  " + row[8].lower().replace(" ", "  ") + "  "
    elif kind == "name_missing": row[9] = ""

def inject_missing(row, missing_rng):
    # Preserve the schema: blank only existing fields, including identifiers and amounts.
    columns = missing_rng.sample(range(len(HEADERS)), missing_rng.randint(1, MAX_MISSING_FIELDS))
    newly_blank = [HEADERS[column] for column in columns if row[column] != ""]
    for column in columns:
        row[column] = ""
    return [HEADERS[column] for column in columns], newly_blank


## Generate raw transactions with deliberate noise

The filename is retained for compatibility; its contents now include multiple products. The transaction-ID suffix is unique before noise injection. No risk score or extra column is added to the CSV.


In [3]:
targets = [OUTPUT_FILE, REPORT_FILE] + ([CLEAN_FILE, QUARANTINE_FILE] if WRITE_CLEAN_OUTPUTS else [])
if not OVERWRITE and any(path.exists() for path in targets):
    raise FileExistsError("An output exists. Set OVERWRITE=True to replace it.")
rng = random.Random(SEED)
noise_rng = random.Random(SEED + 1)
missing_rng = random.Random(SEED + 2)
missing_selected_fields = Counter()
newly_blank_fields = Counter()
generation_counts = Counter()
injected = Counter()
product_counts = Counter()
previous_id = None
started = time.perf_counter()
descriptor, temp_name = tempfile.mkstemp(dir=OUTPUT_DIR, suffix=".csv.tmp")
os.close(descriptor)
temp_path = Path(temp_name)
try:
    with temp_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(HEADERS)
        for seq in range(1, ROW_COUNT + 1):
            row = clean_record(rng, seq)
            product_counts[row[1]] += 1
            noise_selected = noise_rng.random() < NOISE_RATE
            if noise_selected:
                choices = NOISE_TYPES if previous_id else tuple(k for k in NOISE_TYPES if k != "id_duplicate")
                kind = noise_rng.choice(choices)
                inject_noise(row, kind, previous_id, noise_rng)
                injected[kind] += 1
            missing_selected = missing_rng.random() < MISSING_ROW_RATE
            if missing_selected:
                selected, newly_blank = inject_missing(row, missing_rng)
                missing_selected_fields.update(selected)
                newly_blank_fields.update(newly_blank)
                generation_counts["rows_selected_for_missing_injection"] += 1
            generation_counts["rows_selected_for_noise_or_missing"] += int(noise_selected or missing_selected)
            generation_counts["rows_selected_for_both"] += int(noise_selected and missing_selected)
            if row[0]: previous_id = row[0]
            writer.writerow(row)
            if seq % 100_000 == 0: print(f"Written {seq:,} rows")
    os.replace(temp_path, OUTPUT_FILE)
finally:
    temp_path.unlink(missing_ok=True)
print(f"Generated {ROW_COUNT:,} rows in {time.perf_counter() - started:.1f}s")
print(f"Noise-injected rows: {sum(injected.values()):,}; file size: {OUTPUT_FILE.stat().st_size / 1024**2:.1f} MiB")
print("Injected noise counts:", dict(sorted(injected.items())))


Written 100,000 rows
Written 200,000 rows
Written 300,000 rows
Written 400,000 rows
Written 500,000 rows
Written 600,000 rows
Written 700,000 rows
Written 800,000 rows
Written 900,000 rows
Written 1,000,000 rows
Generated 1,000,000 rows in 26.5s
Noise-injected rows: 80,042; file size: 196.7 MiB
Injected noise counts: {'account_missing': 3570, 'account_spacing': 3656, 'amount_grouping': 3671, 'amount_missing': 3647, 'amount_negative': 3729, 'amount_text': 3590, 'country_alpha_hh': 3550, 'country_invalid': 3615, 'country_lowercase': 3707, 'country_missing': 3716, 'country_numeric_97': 3564, 'date_dash': 3700, 'date_impossible': 3726, 'date_missing': 3750, 'date_sentinel': 3614, 'date_slash': 3617, 'id_duplicate': 3557, 'id_missing': 3663, 'name_missing': 3646, 'name_spacing': 3531, 'product_spacing': 3651, 'product_typo': 3572}


## Normalize what is unambiguous; quarantine what is not

The following function never edits the raw CSV. It preserves leading zeros, normalizes approved product aliases, known date formats, name spacing/case, and country case. Invalid or missing required values are rejected rather than guessed. Both raw-name fields remain optional. A transaction-ID date prefix is checked against the normalized value date. Duplicate checks are performed separately using a temporary SQLite index, avoiding a million-entry Python set.

An approved product alias corrects spelling only; it cannot prove that an international ACH is the correct business classification. The quality pass also rejects domestic/cross-border corridor contradictions in the limited scenarios where this generator promises a specific corridor.


In [4]:
ALIASES = {product.casefold(): product for product in PRODUCTS}
ALIASES["internation ach"] = "International ACH"
VALID_DATES = set(DATES)
VALID_COUNTRIES = set(COUNTRIES)
ID_PATTERN = re.compile(r"[0-9]{8}[CS][0-9]{7}")
ACCOUNT_PATTERN = re.compile(r"(?:0[12][0-9]{12}|GB00SYNX[12][0-9]{11}|99[12][0-9]{10}(?:USD|EUR|GBP))")

def normalize_record(original):
    if len(original) != len(HEADERS):
        return list(original), ["field_count"], []
    row = list(original)
    errors, fixes = [], []
    for i in range(len(HEADERS)):
        row[i] = row[i].strip()
    key = row[0]
    if not ID_PATTERN.fullmatch(key) or key[:8] not in VALID_DATES:
        errors.append("invalid_transaction_id")
    product = ALIASES.get(" ".join(row[1].split()).casefold())
    if product is None: errors.append("invalid_product")
    else: row[1] = product
    value = row[2]
    if re.fullmatch(r"[0-9]{1,3}(?:,[0-9]{3})+", value): value = value.replace(",", "")
    if not re.fullmatch(r"[0-9]+", value) or not 1 <= int(value) <= 200_000_000:
        errors.append("invalid_amount")
    else: row[2] = str(int(value))
    value = row[3]
    if re.fullmatch(r"[0-9]{4}([/-])[0-9]{2}\1[0-9]{2}", value):
        value = value.replace("/", "").replace("-", "")
    if value not in VALID_DATES:
        errors.append("invalid_date")
    else:
        row[3] = value
        if ID_PATTERN.fullmatch(key) and key[:8] != value: errors.append("id_date_mismatch")
    for i in (4, 5):
        if not ACCOUNT_PATTERN.fullmatch(row[i]): errors.append(HEADERS[i] + "_invalid")
    for i in (8, 9):
        row[i] = " ".join(row[i].split()).upper()
        if not row[i].startswith("SYNTHETIC "): errors.append(HEADERS[i] + "_invalid")
    for i in (10, 11):
        row[i] = row[i].upper()
        if row[i] not in VALID_COUNTRIES: errors.append(HEADERS[i] + "_invalid")
    if all(row[i] in VALID_COUNTRIES for i in (10, 11)):
        if row[1] in ("Domestic ACH", "P2P Transfer", "Wallet Transfer") and row[10] != row[11]:
            errors.append("domestic_corridor_mismatch")
        if row[1] in ("International ACH", "Cross-Border Remittance") and row[10] == row[11]:
            errors.append("international_corridor_mismatch")
    for i, value in enumerate(row):
        if not value and i not in (6, 7):
            errors.append(HEADERS[i] + "_missing")
    fixes = [HEADERS[i] for i in range(len(HEADERS)) if row[i] != original[i]]
    return row, errors, fixes

# Small in-memory checks verify the handling logic without producing another dataset.
reference = clean_record(random.Random(10), 1)
assert not normalize_record(reference)[1]
for invalid in ("99999999", "20260230", "", "20241231", "20261001"):
    check = reference.copy(); check[3] = invalid
    assert "invalid_date" in normalize_record(check)[1]
for boundary in ("20250101", "20260930"):
    check = reference.copy(); check[3] = boundary; check[0] = boundary + check[0][8:]
    assert not normalize_record(check)[1]
for separator in ("/", "-"):
    check = reference.copy(); day = check[3]
    check[3] = separator.join((day[:4], day[4:6], day[6:]))
    assert normalize_record(check)[0][3] == day and not normalize_record(check)[1]
for column in (10, 11):
    for invalid in ("97", "HH", "ZZ", ""):
        check = reference.copy(); check[column] = invalid
        assert HEADERS[column] + "_invalid" in normalize_record(check)[1]
    check = reference.copy(); check[column] = check[column].lower()
    assert not normalize_record(check)[1]
for column in range(len(HEADERS)):
    check = reference.copy(); check[column] = ""
    errors = normalize_record(check)[1]
    if column in (6, 7):
        assert not errors  # Raw names are optional.
    else:
        assert HEADERS[column] + "_missing" in errors
for kind, expected in (("country_numeric_97", "97"), ("country_alpha_hh", "HH")):
    affected = set()
    for seed in range(20):
        check = reference.copy()
        inject_noise(check, kind, reference[0], random.Random(seed))
        assert len(check) == len(HEADERS)
        affected.update(i for i in (10, 11) if check[i] == expected)
    assert affected == {10, 11}
check = reference.copy()
selected, newly_blank = inject_missing(check, random.Random(20))
assert 1 <= len(selected) <= MAX_MISSING_FIELDS
assert all(check[HEADERS.index(name)] == "" for name in selected)
assert len(check) == 12
print("Date, country-noise, missing-value, optional-field, and 12-column schema checks passed.")


Date, country-noise, missing-value, optional-field, and 12-column schema checks passed.


In [5]:
started = time.perf_counter()
summary = Counter()
issue_counts, normalization_counts = Counter(), Counter()
blank_counts = Counter({header: 0 for header in HEADERS})
country_observed = {HEADERS[i]: Counter({"97": 0, "HH": 0, "ZZ": 0, "": 0}) for i in (10, 11)}
minimum_date = maximum_date = None
with tempfile.TemporaryDirectory() as temporary_dir:
    connection = sqlite3.connect(str(Path(temporary_dir) / "seen.sqlite"))
    connection.execute("CREATE TABLE seen (id TEXT PRIMARY KEY) WITHOUT ROWID")
    streams = []
    try:
        clean_writer = quarantine_writer = None
        if WRITE_CLEAN_OUTPUTS:
            for path in (CLEAN_FILE, QUARANTINE_FILE):
                streams.append(path.open("w", encoding="utf-8-sig", newline=""))
            clean_writer, quarantine_writer = [csv.writer(stream) for stream in streams]
            clean_writer.writerow(HEADERS)
            quarantine_writer.writerow(HEADERS)
        with OUTPUT_FILE.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle, strict=True)
            if next(reader, None) != HEADERS: raise ValueError("Header mismatch")
            for line, original in enumerate(reader, 2):
                row, errors, fixes = normalize_record(original)
                if len(row) != len(HEADERS): raise ValueError(f"Malformed record at CSV line {line}")
                blank_counts.update(HEADERS[i] for i, value in enumerate(original) if value == "")
                if any(value == "" for value in original): summary["rows_with_any_blank"] += 1
                if any(original[i] == "" for i in range(len(HEADERS)) if i not in (6, 7)):
                    summary["rows_with_required_blank"] += 1
                for i in (10, 11):
                    if original[i] in ("97", "HH", "ZZ", ""):
                        country_observed[HEADERS[i]][original[i]] += 1
                if row[0]:
                    cursor = connection.execute("INSERT OR IGNORE INTO seen VALUES (?)", (row[0],))
                    if cursor.rowcount == 0: errors.append("duplicate_transaction_id")
                summary["total_rows"] += 1
                issue_counts.update(errors)
                normalization_counts.update(fixes)
                if fixes: summary["rows_with_normalizations"] += 1
                if errors:
                    summary["quarantined_rows"] += 1
                    if quarantine_writer: quarantine_writer.writerow(original)
                else:
                    summary["accepted_rows"] += 1
                    summary["accepted_after_normalization" if fixes else "accepted_unchanged"] += 1
                    minimum_date = row[3] if minimum_date is None else min(minimum_date, row[3])
                    maximum_date = row[3] if maximum_date is None else max(maximum_date, row[3])
                    if clean_writer: clean_writer.writerow(row)
        if summary["total_rows"] != ROW_COUNT: raise ValueError("Unexpected row count")
        if summary["accepted_rows"] + summary["quarantined_rows"] != ROW_COUNT:
            raise ValueError("Row reconciliation failed")
        if summary["quarantined_rows"] > generation_counts["rows_selected_for_noise_or_missing"]:
            raise ValueError("Unexpected quality errors in unmodified records")
    finally:
        connection.close()
        for stream in streams: stream.close()

report = {
    "seed": SEED, "noise_probability": NOISE_RATE, "expected_rows": ROW_COUNT,
    "schema": HEADERS, "missing_row_probability": MISSING_ROW_RATE,
    "max_missing_fields": MAX_MISSING_FIELDS,
    "generation_counts": dict(generation_counts),
    "missing_injection_selected_fields": dict(missing_selected_fields),
    "newly_blank_fields_from_missing_injection": dict(newly_blank_fields),
    "observed_blank_cells_by_column": dict(blank_counts),
    "observed_country_noise": {name: dict(counts) for name, counts in country_observed.items()},
    "valid_date_start": str(START_DATE), "valid_date_end": str(END_DATE),
    "injected_noise_rows": sum(injected.values()), "injected_noise_counts": dict(injected),
    "products_before_noise": dict(product_counts), "summary": dict(summary),
    "quality_issue_counts": dict(issue_counts), "normalized_field_counts": dict(normalization_counts),
    "accepted_date_min": minimum_date, "accepted_date_max": maximum_date,
    "clean_outputs_written": WRITE_CLEAN_OUTPUTS,
    "notes": "Issue counts can overlap. Raw data is never silently repaired. No financial-crime labels are inferred."
}
REPORT_FILE.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"Full quality pass completed in {time.perf_counter() - started:.1f}s")
print(f"Raw CSV: {OUTPUT_FILE}")
print(f"Quality report: {REPORT_FILE}")
if WRITE_CLEAN_OUTPUTS: print(f"Clean: {CLEAN_FILE}\nQuarantine: {QUARANTINE_FILE}")


{
  "seed": 20260930,
  "noise_probability": 0.08,
  "expected_rows": 1000000,
  "schema": [
    "trxn_id",
    "lvl_2_product_txt",
    "amount",
    "value_date",
    "orig_raw_account_num",
    "bene_raw_account_num",
    "orig_raw_name",
    "bene_raw_name",
    "orig_clean_name",
    "bene_clean_name",
    "orig_cntry_code",
    "bene_cntry_code"
  ],
  "missing_row_probability": 0.03,
  "max_missing_fields": 3,
  "generation_counts": {
    "rows_selected_for_noise_or_missing": 107839,
    "rows_selected_for_both": 2448,
    "rows_selected_for_missing_injection": 30245
  },
  "missing_injection_selected_fields": {
    "orig_raw_account_num": 5182,
    "orig_clean_name": 5042,
    "orig_cntry_code": 5155,
    "bene_clean_name": 5127,
    "amount": 4939,
    "bene_cntry_code": 4966,
    "trxn_id": 5069,
    "value_date": 4985,
    "bene_raw_name": 5063,
    "bene_raw_account_num": 4948,
    "lvl_2_product_txt": 5123,
    "orig_raw_name": 5067
  },
  "newly_blank_fields_from_missing_

## Interpreting results

The report separates rows selected for noise or missing-value injection from rows actually rejected. It reports observed blanks for every original column and observed `97`, `HH`, `ZZ`, and blank values in both country columns. Optional raw names are already blank in some clean baseline records, so `rows_with_any_blank` is not the same as the deliberate missing-value injection rate. A selected field may already be blank; both selected-field counts and newly-created blank counts are included. The two independent injection probabilities can overlap and must not simply be added. Mixed-format dates, lowercase codes, and approved product aliases may be repairable. Invalid dates, missing identifiers, and ambiguous amounts are not silently imputed. Error counts can exceed rejected rows because one row can fail several checks.

To produce accepted and rejected files, set WRITE_CLEAN_OUTPUTS=True and rerun. The accepted CSV retains the exact 12-column schema. The quarantine CSV preserves the same original 12 columns and the original values. Aggregate rejection reasons are stored separately in the JSON quality report, not as added dataset columns. These optional files require additional disk space; only use them after the full quality cell completes. The raw CSV remains unchanged. These controls address data quality, not AML detection or regulatory reporting requirements.
